<div style="background: linear-gradient(135deg, #fdfbfb 0%, #ebedee 100%); border-radius: 10px; border-left: 6px solid #ff9900; box-shadow: 0 3px 6px rgba(0,0,0,0.06); width: 100%; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; overflow: hidden;">
  <table style="border: none; width: 100%; margin: 0; border-collapse: collapse; background: transparent;">
    <tr style="border: none; background: transparent;">
      <td style="border: none; width: 100px; text-align: center; vertical-align: middle; padding: 14px 0 14px 15px; background: transparent;">
        <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
      </td>
      <td style="border: none; vertical-align: middle; padding: 14px 15px; background: transparent;">
        <div style="color: #555555; font-size: 13px; font-weight: 500; margin-bottom: 2px;">Excerpt from the Amazon book</div>
        <div style="font-family: 'Arial Black', Impact, sans-serif; font-size: 18px; font-weight: 900; text-transform: uppercase; letter-spacing: 1.5px; color: #111111; margin: 6px 0 10px 0; line-height: 1.2;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</div>
        <div>
          <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: none; font-weight: 600; font-size: 13px; display: inline-block;">🛒 Buy on Amazon &rarr;</a>
        </div>
      </td>
    </tr>
  </table>
</div>

# **For this notebook to work, you must execute ALL the cells in the order in which they appear.**


# **Getting data to our JVM**

In [1]:
!wget -O 2019.csv https://raw.githubusercontent.com/sansalamanca/shining-path_DP700/main/data/2019.csv

--2026-11-14 21:31:35--  https://raw.githubusercontent.com/sansalamanca/shining-path_DP700/main/data/2019.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 123882 (121K) [text/plain]
Saving to: ‘2019.csv’

2019.csv            100%[===================>] 120.98K  --.-KB/s    in 0.02s   

2026-11-14 21:31:36 (7.03 MB/s) - ‘2019.csv’ saved [123882/123882]



In [2]:

# Create a Spark session with a custom name.
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("Colab_Session_11") \
.getOrCreate()


# **Observe the data frame we are working with**

In [5]:

from pyspark.sql.types import *

orderSchema = StructType([
   StructField("SalesOrderNumber", StringType()),
   StructField("SalesOrderLineNumber", IntegerType()),
   StructField("OrderDate", DateType()),
   StructField("CustomerName", StringType()),
   StructField("Email", StringType()),
   StructField("Item", StringType()),
   StructField("Quantity", IntegerType()),
   StructField("UnitPrice", FloatType()),
   StructField("Tax", FloatType())
])


df = spark.read.format("csv").schema(orderSchema).load("2019.csv")
df.show()     # In Fabric, use display(df)







+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|         

# **Grouping and aggregating data**

In [6]:
# Group and aggregate data in a DF. Here, we group by product name
# and count how many units of each product were sold.
productSales = df.select("Item", "Quantity").groupBy("Item").sum().withColumnRenamed("sum(Quantity)", "Unidades_Vendidas")
productSales.show()      # In Fabric use:     display(productSales)


+--------------------+-----------------+
|                Item|Unidades_Vendidas|
+--------------------+-----------------+
|Mountain-100 Silv...|               21|
|Mountain-100 Silv...|               28|
|Mountain-100 Silv...|               39|
|    Road-650 Red, 52|               10|
|Mountain-100 Blac...|               35|
|Mountain-100 Blac...|               23|
|  Road-650 Black, 60|               15|
|    Road-150 Red, 62|              197|
|    Road-650 Red, 48|               13|
|  Road-650 Black, 44|               14|
|    Road-150 Red, 44|              157|
|    Road-650 Red, 58|                9|
|  Road-650 Black, 62|               15|
|    Road-650 Red, 60|               13|
|    Road-650 Red, 62|                4|
|  Road-650 Black, 48|                4|
|  Road-650 Black, 52|               20|
|Mountain-100 Blac...|               24|
|    Road-150 Red, 48|              179|
|    Road-150 Red, 52|              155|
+--------------------+-----------------+
only showing top

# **Query using the SQL engine**



In [7]:
# Group by year and count the number of sales

from pyspark.sql.functions import *
yearlySales = df.select(year(col("OrderDate")).alias("Year")).groupBy("Year").count().orderBy("Year")
yearlySales.show()    # En Fabric use:   display(yearlySales)

+----+-----+
|Year|count|
+----+-----+
|2019| 1201|
+----+-----+

